In [1]:
with open("src/ingredient_db.py") as f:
    print(f.read())


import pandas as pd

def load_ingredient_db(path='paulaschoice_us_ingredients_clean.csv'):
    return pd.read_csv(path)

def get_known_terms(db):
    names = set(db['Name'].str.lower())
    return list(names)



In [2]:
with open("src/fuzzy_matcher.py") as f:
    print(f.read())


import pandas as pd
from rapidfuzz import process, fuzz
from .ingredient_db import load_ingredient_db, get_known_terms

db = load_ingredient_db()
known_terms = get_known_terms(db)

def auto_correct_ocr(ocr_text, threshold=80):
    match = process.extractOne(ocr_text.lower().strip(), known_terms, scorer=fuzz.token_sort_ratio)
    if match and match[1] >= threshold:
        return match[0]
    return ocr_text.lower().strip()

def match_ingredient(ingredient, threshold=80):
    corrected = auto_correct_ocr(ingredient, threshold)
    match = process.extractOne(corrected, db['Name'], scorer=fuzz.token_sort_ratio)
    if match and match[1] >= threshold:
        row = db[db['Name'] == match[0]].iloc[0]
        return {
            'Input': ingredient,
            'Corrected': corrected,
            'Matched Name': row['Name'],
            'Description': row['Description'],
            'Rating': row['Rating'],
            'Score': match[1]
        }
    return {
        'Input': ingredient,
 

In [3]:
with open("src/image_utils.py") as f:
    print(f.read())

import cv2

def preprocess_image(image_path):

    img = cv2.imread(image_path)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    gray = cv2.fastNlMeansDenoising(gray)

    thresh = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        11
    )

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2,2))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    return thresh


In [4]:
with open("src/ocr_processor.py") as f:
    print(f.read())


import pytesseract
import pandas as pd
from rapidfuzz import process, fuzz
from .image_utils import preprocess_image
from .fuzzy_matcher import match_ingredient

def extract_ingredients(text):
    lines = text.split(',')
    cleaned = []
    for line in lines:
        line = line.lower().strip()
        for token in ["(and)", "[and]", "-", "ingredients:"]:
            line = line.replace(token, '')
        if line:
            cleaned.append(line)
    return cleaned

def process_image(image_path, threshold=80):
    img = preprocess_image(image_path)
    ocr_text = pytesseract.image_to_string(img)
    ingredients = extract_ingredients(ocr_text)
    results = [match_ingredient(ing, threshold) for ing in ingredients]
    df_results = pd.DataFrame(results)
    unmatched = df_results[df_results['Matched Name'].isna()]
    if not unmatched.empty:
        unmatched.to_csv("unmatched_ingredients.csv", mode='a', index=False, header=False)
        print(f"⚠️ Logged {len(unmatched)} unmatched in

In [5]:
from src.ingredient_db import load_ingredient_db

db = load_ingredient_db()

print(db.shape)

(26018, 4)


In [6]:
from src.fuzzy_matcher import bulk_process

test = [
    "ethylhexglcerin",
    "sodium hualronate",
    "xyzrandomchemical"
]

df = bulk_process(test)

print(df)

⚠️ Logged 1 unmatched ingredients to 'unmatched_ingredients.csv'
               Input           Corrected        Matched Name  \
0    ethylhexglcerin  ethylhexylglycerin  Ethylhexylglycerin   
1  sodium hualronate  sodium hyaluronate  Sodium Hyaluronate   
2  xyzrandomchemical   xyzrandomchemical                None   

                                         Description Rating      Score  
0  Skin-softening agent with hydration boosting p...   good  94.444444  
1  The more bio-available, salt form of hyaluroni...   best  88.888889  
2                                               None   None  52.941176  


In [7]:
from src.ocr_processor import process_image

df = process_image("label1.png")

print(df.head(20))

⚠️ Logged 20 unmatched ingredients to 'unmatched_ingredients.csv'
                                                Input  \
0                  ireienits aque gydlqaaniasttoxerna   
1                                           clycarine   
2                    cepnic/\ncayarie tine hycartidha   
3                                       dninetlnicone   
4                                      le@lnexe@ecene   
5   celvuatcoon\nclyvaaryl sie@araue (ered) aeg: (...   
6          brutiyreg@eriawiay\nparlian (tinal) butiar   
7   wywiliglueosiete (eral anlayairaqdicl lane\nil...   
8                                         innaamatele   
9   clycarin (ancl) agua (ani euiyneme clea:\nlamd...   
10       ree@a te /atare\n{saraaanny) fruit exdiracti   
11                             saghwin inbyeluiremate   
12  riulawic onan hee.\ninmoy rubs [ (chourloanny)...   
13                                         peminanell   
14  cai@an av nov ll 2 ws\ntet oilsesduloxaney rol...   
15        acnleieec to

In [8]:
from src.image_utils import preprocess_image
import pytesseract

img = preprocess_image("label1.png")

ocr_text = pytesseract.image_to_string(img)

print(ocr_text)

IREIENITS aque Gydlqaaniasttoxerna, Clycarine, Cepnic/-
Cayarie Tine hycartidha, Dninetlnicone, le@lnexe@ecene, CELVUATCOON
Clyvaaryl Sie@araue (ered) AEG: (OW Kiearrete,, Brutiyreg@eriawiay
Parlian (tinal) Butiar, wyWiliglueosiete (eral Anlayairaqdicl lane
ilbiall, INNAAMATELE, Clycarin (ancl) Agua (ani Euiyneme Clea:
lamdl] Carlaomear (aml) PRolysoraatie 20) (ina) Pelinii@yl

Timaaudasd (Enel) Paliniiow Tewapaaiets: 7, rEe@a Te /ATarE
{Saraaanny) Fruit Exdiracti,, Saghwin Inbyeluiremate, Riulawic onan HEE.
Inmoy rubs [ (Chourloanny) Se ae! ONL, Peminanell, Cai@an Av NOV LL 2 Ws
TET oilsesduloxaney RolsoubarcoU WAT MONO UIAGRY LOM
shenadnyaurai@AP Capalymen,, AcnleieeC TO) SAD Ney AA SLE
Crasqaalyumar, Alemi@nn, Disewhurnn ENA, Newayanarl ACEH
Pnem@rnqyadmermell (vnal] EVA UMaa a eErT

Saturn nyelrodela, FP

=< .
{eo
Ee,



In [1]:
from src.image_utils import preprocess_image
import pytesseract

img = preprocess_image("label1.png")

ocr_text = pytesseract.image_to_string(img)

print(ocr_text)

NOREDIENTS Agua, Cyclopentasiloxane, Glycerine, Caprylic/-
| pride, De methic one, Isohexadecane, Cetyl Alcohol,
rate (and! PEG 109 Stearate, Butyrospermum
3) Butter, \Wwiitvlgt hicowde (and] Anhydroxylitol (and)
ing ide, Ciye erin (and) Aqua [and] Butylene Glycol
nel ‘land’ Poiysarbate 20 tand] Palmitoyl
Bari) Parnioyt fetrapeptide-7, Fragaria Ananassa
tier it Ext) act, Sodium Hyaluronate, Rubus Cnamae-
gerry] Gord Oil, Panthenol, Cetearyl Aicons. Pasy-
—_ a lysorbate 80, Ammon TALIS.




In [2]:
import os

print(os.listdir("src"))

['ingredient_extractor.ipynb', 'image_utils.py', '__init__.py', '__pycache__', 'ocr_processor.py', 'ingredient_db.py', 'fuzzy_matcher.py', '.ipynb_checkpoints']


In [3]:
from src.extract_and_match import extract_and_match

ocr_text = """
NOREDIENTS Agua, Cyclopentasiloxane, Glycerine, Caprylic/-
| pride, De methic one, Isohexadecane, Cetyl Alcohol,
rate (and! PEG 109 Stearate, Butyrospermum
3) Butter
"""

df = extract_and_match(ocr_text)

print(df[["Input","Matched Name","Score"]])

                Input        Matched Name      Score
0  cyclopentasiloxane  Cyclopentasiloxane  94.444444
1           glycerine           Glycerine  88.888889
2       de methic one         Dimethicone  90.909091
3       isohexadecane       Isohexadecane  92.307692
4       cetyl alcohol       Cetyl Alcohol  84.615385
5            3 butter              Butter  83.333333


In [4]:
from src.image_utils import preprocess_image
import pytesseract

img = preprocess_image("label1.png")

ocr_text = pytesseract.image_to_string(img)

In [5]:
from src.extract_and_match import extract_and_match

df = extract_and_match(ocr_text)

print(df[["Input","Matched Name","Score"]])
print("\nRecovered:", len(df))

                Input        Matched Name      Score
0  cyclopentasiloxane  Cyclopentasiloxane  94.444444
1           glycerine           Glycerine  88.888889
2       de methic one         Dimethicone  90.909091
3       isohexadecane       Isohexadecane  92.307692
4       cetyl alcohol       Cetyl Alcohol  84.615385
5            3 butter              Butter  83.333333
6     butylene glycol     Butylene Glycol  86.666667
7  sodium hyaluronate  Sodium Hyaluronate  88.888889
8           panthenol           Panthenol  88.888889
9      a lysorbate 80      Polysorbate 80  92.857143

Recovered: 10


In [6]:
from src.ocr_processor_v2 import process_image_v2

df = process_image_v2("label1.png")

print(df[["Input","Matched Name","Score"]])

print("\nRecovered:", len(df))

                Input        Matched Name      Score
0  cyclopentasiloxane  Cyclopentasiloxane  94.444444
1           glycerine           Glycerine  88.888889
2       de methic one         Dimethicone  90.909091
3       isohexadecane       Isohexadecane  92.307692
4       cetyl alcohol       Cetyl Alcohol  84.615385
5            3 butter              Butter  83.333333
6     butylene glycol     Butylene Glycol  86.666667
7  sodium hyaluronate  Sodium Hyaluronate  88.888889
8           panthenol           Panthenol  88.888889
9      a lysorbate 80      Polysorbate 80  92.857143

Recovered: 10


## testing on 19th morning

In [7]:
import os

print(os.listdir("src"))

['ocr_processor_v2.py', 'image_utils.py', 'extract_and_match.py', '.DS_Store', '__init__.py', '__pycache__', 'ocr_processor.py', 'ingredient_db.py', 'fuzzy_matcher.py', 'ingredient_extractor.py', '.ipynb_checkpoints']


In [9]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 9.2 MB/s eta 0:00:009.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.5/797.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 5.3 MB/s eta 0:00:00m eta 0:00:010:0101
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.9.0
    Uninstalling typing_extensions-4.9.0:
      Successfully uninstalled typing_extensions-4.9.0
  Attempting uninstall: tenacity
    Found existing installation: tenacity 8.0.1
    Uninstalling tenacity-8.0.1:
      Successfully uninstalled tenacity-8.0.1
  Attempting uninstall: itsdangerous
    Found existing installation: itsdangerous 2.0.1
    Uninstalling itsdangerous-2.0.1:
      Successfully uninstalled itsdangerous-2.0.1
  Attempting uninstall: anyio
    Found existing installation: anyio 3.5.0
    Uninstalling anyio-3.5.0:
      Successfully uninstalled anyio-3.5.0
ERROR: pip's dependency resolver does not curre


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [3]:
!streamlit --version

Streamlit, version 1.58.0
